# Notebook 3: Context-Aware Pipeline
 agentic-civic-resolution-app

 Goal: Wire Delta memory into the agent pipeline.
 Every complaint is processed with full awareness of past incidents at the same location.

 Flow:
 ```
 complaint_text
     │
     ▼
 [memory.get_location_context()]   ← past incidents at this location
     │
     ▼
 [Agent 1: Classify]               ← category, subcategory
     │
     ▼
 [memory.check_recurring()]        ← is this a repeat at same spot?
     │
     ▼
 [Agent 2: Score]                  ← severity boosted +1 if chronic
     │
     ▼
 [Agent 3: Route]                  ← dept, officer tier, action plan
     │
     ▼
 [memory.save_ticket()]            ← persist to Delta
     │
     ▼
 CivicOpsTicket (context-aware)
```


In [0]:
%pip install databricks-langchain mlflow pydantic --quiet

#dbutils.library.restartPython()

## 0. Load Notebooks

In [0]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
print(ctx.notebookPath().get())

In [0]:
%run /Workspace/Users/gaurab.dawn@anz.com/agentic-civic-resolution-app/02_agents/agent_01_classifier


In [0]:
%run /Workspace/Users/gaurab.dawn@anz.com/agentic-civic-resolution-app/02_agents/agent_02_severity

In [0]:
%run /Workspace/Users/gaurab.dawn@anz.com/agentic-civic-resolution-app/02_agents/agent_03_router

In [0]:
%run /Users/gaurab.dawn@anz.com/agentic-civic-resolution-app/03_memory_layer/02_memory_manager

## 1. Context-Aware Orchestrator

In [0]:
import time, uuid, re
from datetime import datetime, timedelta, timezone

SLA_HOURS       = {5: 4, 4: 24, 3: 72, 2: 168, 1: 720}
PRIORITY_LABELS = {5: "Critical", 4: "High", 3: "Medium", 2: "Low", 1: "Minimal"}

In [0]:
class ContextAwareOrchestrator:

    def __init__(self):
        self.classifier = classifier_agent
        self.scorer     = severity_agent
        self.router     = router_agent
        self.memory     = memory

    def process(self, complaint_text: str) -> dict:
        start         = time.time()
        location_hint = self._extract_location(complaint_text)

        # ── Step 0: Memory Lookup ─────────────────────────────────────────────
        past_incidents = self.memory.get_location_context(location_hint, limit=3)

        # ── Step 1: Classify ──────────────────────────────────────────────────
        cls = self.classifier.classify(complaint_text)

        # ── Step 2: Check Recurring ───────────────────────────────────────────
        recurring_info = self.memory.check_recurring(location_hint, cls.category)

        # ── Step 3: Score ─────────────────────────────────────────────────────
        sev = self.scorer.score(
            complaint_text,
            category=cls.category,
            subcategory=cls.subcategory,
        )

        # Boost severity +1 for chronic locations
        if recurring_info.get("is_chronic") and sev.severity_score < 5:
            original            = sev.severity_score
            sev.severity_score  = sev.severity_score + 1
            sev.sla_hours       = SLA_HOURS[sev.severity_score]
            sev.priority_label  = PRIORITY_LABELS[sev.severity_score]
            print(f"  ⚡ Chronic boost: {original} → {sev.severity_score} ({sev.priority_label})")

        # ── Step 4: Route ─────────────────────────────────────────────────────
        route = self.router.route(
            complaint_text=complaint_text,
            category=cls.category,
            subcategory=cls.subcategory,
            severity_score=sev.severity_score,
            health_risk=sev.health_risk,
            infrastructure_risk=sev.infrastructure_risk,
        )

        elapsed_ms = int((time.time() - start) * 1000)

        # ── Step 5: Assemble Ticket ───────────────────────────────────────────
        ticket = {
            "ticket_id":                  f"TKT-{uuid.uuid4().hex[:8].upper()}",
            "raw_complaint":              complaint_text,
            "category":                   cls.category,
            "subcategory":                cls.subcategory,
            "classification_confidence":  cls.confidence,
            "severity_score":             sev.severity_score,
            "priority_label":             sev.priority_label,
            "sla_hours":                  sev.sla_hours,
            "sla_deadline":               datetime.now(timezone.utc) + timedelta(hours=sev.sla_hours),
            "health_risk":                sev.health_risk,
            "infrastructure_risk":        sev.infrastructure_risk,
            "affected_estimate":          sev.affected_estimate,
            "dept_code":                  route.dept_code,
            "dept_name":                  route.dept_name,
            "dept_contact":               route.dept_contact,
            "officer_tier":               route.officer_tier,
            "escalate":                   route.escalate,
            "escalation_dept":            route.escalation_dept,
            "action_plan":                route.action_plan,
            "field_visit_needed":         route.field_visit_needed,
            "notify_citizen":             route.notify_citizen,
            "processing_ms":              elapsed_ms,
            # Memory-enriched
            "is_recurring":               recurring_info.get("is_recurring", False),
            "occurrence_count":           recurring_info.get("occurrence_count", 0),
            "is_chronic":                 recurring_info.get("is_chronic", False),
            "past_incidents_count":       len(past_incidents),
        }

        # ── Step 6: Persist ───────────────────────────────────────────────────
        self.memory.save_ticket(ticket)

        return ticket

    def print_summary(self, ticket: dict):
        chronic  = " 🔴 CHRONIC" if ticket.get("is_chronic") else ""
        recur    = f" (#{ticket['occurrence_count']} at this location)" if ticket.get("is_recurring") else ""
        print(f"""
╔══ {ticket['ticket_id']} ══════════════════════════════════
║ Complaint  : {ticket['raw_complaint']}
║ Category   : {ticket['category']} > {ticket['subcategory']}{recur}{chronic}
║ Severity   : {ticket['severity_score']}/5 — {ticket['priority_label']}
║ SLA        : {ticket['sla_hours']}h deadline
║ Department : [{ticket['dept_code']}] {ticket['dept_name']}
║ Escalate   : {'YES → ' + (ticket['escalation_dept'] or '') if ticket['escalate'] else 'No'}
║ Past incidents at location: {ticket['past_incidents_count']}
╚{'═' * 54}""")

    @staticmethod
    def _extract_location(text: str) -> str:
        for p in [r'near\s+(.+?)(?:\.|,|$)', r'at\s+(.+?)(?:\.|,|$)', r'on\s+(.+?)(?:\.|,|$)']:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                return m.group(1).strip()
        return text[:64]


In [0]:
orchestrator = ContextAwareOrchestrator()
print("✓ Context-Aware Orchestrator ready.")

## 2. Demo — Memory in Action

In [0]:
# 3 complaints at same location → triggers chronic flag on 3rd
complaints = [
    "Garbage overflow near Whitefield junction",
    "Garbage dumped again near Whitefield junction — not cleared",
    "Garbage overflow near Whitefield junction still piling up after 3 days",
    "Exposed live wires near MG Road metro station",
    "Water pipeline burst on Brigade Road — road flooding since morning",
]

tickets = []
for c in complaints:
    print(f"\nProcessing: {c}")
    t = orchestrator.process(c)
    orchestrator.print_summary(t)
    tickets.append(t)

## 3. Memory Queries

In [0]:
# Chronic locations
print("=== Chronic Problem Locations ===")
for loc in memory.get_chronic_locations():
    print(f"  📍 {loc['location_key']} | {loc['complaint_type']} | "
          f"occurrences={loc['occurrence_count']} | max_severity={loc['max_severity']}")

# COMMAND ----------

# SLA breached
print("=== SLA Breached Tickets ===")
breached = memory.get_sla_breached()
print(f"  {len(breached)} ticket(s) past SLA deadline.")
for t in breached:
    print(f"  ⚠ {t['ticket_id']} | {t['category']} | {t['hrs_overdue']}h overdue")

# COMMAND ----------

# Status updates
print("=== Updating Statuses ===")
for ticket in tickets[:2]:
    r = memory.update_status(
        ticket["ticket_id"], "IN_PROGRESS",
        changed_by="field_officer_01",
        notes="Team dispatched"
    )
    print(f"  {r['ticket_id']} → {r['new_status']} | SLA breached: {r['sla_breached']}")

## 4. Display Memory Tables

In [0]:
print("=== complaint_history ===")
display(spark.table("civicops.memory.complaint_history")
        .select("ticket_id","category","severity_score","priority_label","status","dept_code","escalate","is_chronic" if "is_chronic" in spark.table("civicops.memory.complaint_history").columns else "location_hint")
        .orderBy("created_at"))


print("=== recurring_complaints ===")
display(spark.table("civicops.memory.recurring_complaints"))


print("=== sla_status_history ===")
display(spark.table("civicops.memory.sla_status_history"))
